# 🔧 DeepHermes 3 — QLoRA Fine-Tune on Colab T4

**Model:** `NousResearch/DeepHermes-3-Llama-3-8B-Preview`  
**Method:** QLoRA (4-bit NF4) via Unsloth  
**GPU:** T4 (16 GB VRAM) — Colab Free Tier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghassan-gaidi/finetune-lab/blob/main/notebooks/lora/example_qlora_colab.ipynb)

---
## What This Does

1. Loads DeepHermes 3 8B in 4-bit — fits in ~5.5 GB VRAM
2. Adds LoRA adapters on all attention + feed-forward layers
3. Trains on a curated **agentic dataset mix** (tool calling + reasoning traces)
4. Exports to GGUF `q4_k_m` and pushes to Hugging Face Hub

Requires ~12 GB free disk in Colab runtime for the base model + outputs.

---
## Setup

### 1a. HF Token (recommended: Colab Secrets)

Set your Hugging Face token as a **Colab Secret** instead of hardcoding:
- Left sidebar → 🔑 Secrets → `HF_TOKEN`
- If you skip this, hardcode below (not recommended for shared notebooks).

In [ ]:
import os
import torch

# Try Colab Secrets first, fall back to env var
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded from Colab Secrets")
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or hardcode below"
os.environ["WANDB_DISABLED"] = "true"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

### 1b. Install Unsloth (GPU-capability aware)

In [ ]:
import torch

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"GPU: {torch.cuda.get_device_name()}  |  Capability: {major}.{minor}")
else:
    raise RuntimeError("No GPU detected - go to Runtime > Change runtime type > T4 GPU")

if major >= 8:
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install --no-deps "xformers<0.0.27" trl peft transformers accelerate

print("Dependencies installed")

---
## Load Model in 4-bit

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = torch.float16
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="NousResearch/DeepHermes-3-Llama-3-8B-Preview",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=os.environ["HF_TOKEN"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print(f"Model loaded. Trainable params: {model.num_parameters(only_trainable=True):,}")

---
## Dataset - Agentic Data Mix

Loads from the **best available agentic datasets** and mixes them into one
training set. Edit `DATASET_MIX` below to change the blend.

### Default Mix

| Dataset | Config | Rows | Weight |
|---------|--------|------|--------|
| `lambda/hermes-agent-reasoning-traces` | kimi | 7,646 | 1.0x |
| `lambda/hermes-agent-reasoning-traces` | glm-5.1 | 7,055 | 1.0x |
| `DJLougen/hermes-agent-traces-filtered` | - | 3,679 | 1.5x |
| `NousResearch/hermes-function-calling-v1` | func_calling | 8,372 | 0.5x |

> Full catalog at [`data/datasets.md`](https://github.com/ghassan-gaidi/finetune-lab/blob/main/data/datasets.md)

In [ ]:
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import get_chat_template

# Apply Llama-3 chat template with ShareGPT field mapping
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3",
    mapping={"role": "from", "content": "value",
             "user": "human", "assistant": "gpt"},
)

# ---------------------------------------------
# Dataset Mix - edit this to change your blend
# ---------------------------------------------
# (hf_path, config_or_None, max_rows_or_0_for_all, weight)
DATASET_MIX = [
    ("lambda/hermes-agent-reasoning-traces", "kimi",     0, 1.0),
    ("lambda/hermes-agent-reasoning-traces", "glm-5.1",  0, 1.0),
    ("DJLougen/hermes-agent-traces-filtered", None,      0, 1.5),
    ("NousResearch/hermes-function-calling-v1", "func_calling", 0, 0.5),
]
# ---------------------------------------------

def format_sharegpt(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

def load_and_format(path, config, limit):
    if config:
        raw = load_dataset(path, config, split="train", token=os.environ["HF_TOKEN"])
    else:
        raw = load_dataset(path, split="train", token=os.environ["HF_TOKEN"])
    if limit and limit < len(raw):
        raw = raw.select(range(limit))
        print(f"  {path}/{config or '-'}: {len(raw)} rows")
    else:
        print(f"  {path}/{config or '-'}: {len(raw)} rows")
    if "conversations" in raw.column_names:
        return raw.map(format_sharegpt, batched=True)
    elif "messages" in raw.column_names:
        return raw.rename_column("messages", "text")
    else:
        raise ValueError(f"Unknown schema for {path}. Columns: {raw.column_names}")

print("=" * 56)
print("  Loading agentic dataset mix")
print("=" * 56)

all_datasets = []
for ds_path, config, limit, weight in DATASET_MIX:
    ds = load_and_format(ds_path, config, limit)
    count = len(ds)
    if weight != 1.0 and count > 0:
        oversample = int(count * weight)
        if oversample > count:
            ds = concatenate_datasets([ds] * (oversample // count + 1))
            ds = ds.select(range(oversample))
            print(f"    weighted {weight}x -> {len(ds)} rows")
    all_datasets.append(ds)

import random
random.shuffle(all_datasets)
dataset = concatenate_datasets(all_datasets)

print(f"\nTotal: {len(dataset)} examples across {len(DATASET_MIX)} datasets")
print(f"  First example:\n{dataset[0]['text'][:200]}...")

---
## Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_strategy="steps",
        save_steps=20,
    ),
)

trainer_stats = trainer.train()
print(f"Training complete - loss: {trainer_stats.metrics['train_loss']:.4f}")

---
## Export & Push to HF Hub

Merges LoRA weights into the base model, converts to GGUF `q4_k_m`.

In [ ]:
HF_REPO_PATH = "leofalco/deep-hermes-Le0Fvlc0"

print(f"Merging adapters, converting to GGUF q4_k_m, pushing to {HF_REPO_PATH}...")

model.push_to_hub_gguf(
    repo_id=HF_REPO_PATH,
    tokenizer=tokenizer,
    quantization_method="q4_k_m",
    token=os.environ["HF_TOKEN"],
)

print(f"Done! Model live at https://huggingface.co/{HF_REPO_PATH}")

---
## Test the Exported Model (in llama.cpp)

After the push completes, download the GGUF and run inference:

In [ ]:
!pip install llama-cpp-python -q
!wget -q https://huggingface.co/leofalco/deep-hermes-Le0Fvlc0/resolve/main/deep-hermes-Le0Fvlc0-unsloth.Q4_K_M.gguf -O model.gguf

from llama_cpp import Llama
llm = Llama(model_path="./model.gguf", n_gpu_layers=-1)
out = llm("<|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\n", max_tokens=128)
print(out['choices'][0]['text'])